# 📊 Data Overview

Import python functions from [api_request.py](api_request.py) file and load data to pandas 🐼

In [53]:
#import importlib
#import api_request # w razie gdyby jupter nie odświeżył pliku .py
#importlib.reload(api_request)

In [54]:
import pandas as pd
from api_request import get_all_top_artists, get_all_top_songs

In [55]:
df_artists = get_all_top_artists()

Page number: 1, Number of rows: 1000
Page number: 2, Number of rows: 1000
Page number: 3, Number of rows: 1000
Page number: 4, Number of rows: 1000
Page number: 5, Number of rows: 1000
Page number: 6, Number of rows: 1000
Page number: 7, Number of rows: 1000
Page number: 8, Number of rows: 1000
Page number: 9, Number of rows: 1000
Page number: 10, Number of rows: 1000
🔴 No more pages
✅ Artits dataset was loaded


In [56]:
print("Total number of rows: ", len(df_artists))

Total number of rows:  10000


In [57]:
df_artists.head()

,name,listeners,playcount
0,PinkPantheress,2996088,314039944
1,The Weeknd,5262970,1105608702
2,Kendrick Lamar,5052099,1018810132
3,Radiohead,8183356,1357535268
4,Tame Impala,4273046,406863139


In [58]:
df_artists.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   name       10000 non-null  object
 1   listeners  10000 non-null  int64 
 2   playcount  10000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 234.5+ KB


In [59]:
df_artists.describe()

,listeners,playcount
count,1.000000e+04,1.000000e+04
mean,6.140104e+05,2.370855e+07
std,8.277772e+05,8.330108e+07
min,4.979000e+03,1.622200e+04
25%,1.581428e+05,2.876915e+06
50%,3.229850e+05,6.559678e+06
75%,7.150865e+05,1.744599e+07
max,9.056027e+06,3.635182e+09


In [60]:
df_songs = get_all_top_songs()

print("Total number of rows: ", len(df_songs))

Page number: 1, Number of rows: 1000
Page number: 2, Number of rows: 1000
Page number: 3, Number of rows: 1000
Page number: 4, Number of rows: 1000
Page number: 5, Number of rows: 1000
Page number: 6, Number of rows: 1000
Page number: 7, Number of rows: 1000
Page number: 8, Number of rows: 1000
Page number: 9, Number of rows: 1000
Page number: 10, Number of rows: 999
🔴 No more pages
✅ Songs dataset was loaded
Total number of rows:  9999


In [61]:
df_songs.head()

,artist,duration,listeners,playcount,name
0,PinkPantheress,176,1034823,15257145,Stateside + Zara Larsson
1,BTS,189,290355,6340978,Body to Body
2,BTS,159,281469,16368994,Swim
3,BTS,182,270237,5813931,Hooligan
4,BTS,180,263292,5716054,FYA


## 🧹 Simple Data Cleanup 

In [ ]:
# remove duplicates 

df_songs.drop_duplicates(inplace=True, subset=['artist', 'name', 'duration'])
df_artists.drop_duplicates(inplace=True, subset=['name', 'listeners'])

# although there are no duplicates in db as far as i can tell

# 🐘 Load Data to SQL Database (PostgreSQL)

Load data from pandas to PostgreSQL

## 🔴 CREATE music_DB IN POSTGRESQL FIRST!

You can do that manually in pgAdmin4 or run terminal and type 2 commands below:
```
psql -U postgres
```
AND THEN:
```
CREATE DATABASE music_db;
```

In [63]:
# PostgreSQL DB connection (change variables if needed - especially db_user and password)

from sqlalchemy import create_engine

DB_USER = "postgres"
DB_PASSWORD = "Password"
DB_HOST = "localhost" 
DB_PORT = "5432"
DB_NAME = "music_db"

engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [64]:
# delete all objects in database first just in case

from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("DROP SCHEMA public CASCADE"))
    conn.execute(text("CREATE SCHEMA public"))

In [65]:
df_artists.to_sql(
    name="top_artists",
    con=engine,
    if_exists="replace",  # lub append
    index=False
)

1000

In [66]:
df_songs.to_sql(
    name="top_songs",
    con=engine,
    if_exists="replace",
    index=True
)

999

In [ ]:
# let's check whether SQL works as planned

test1 = pd.read_sql("SELECT COUNT(*) AS test1 FROM top_songs", engine)
test2 = pd.read_sql("SELECT COUNT(*) AS test2 FROM top_artists", engine)

test1

,count
0,9999


In [68]:
test2

,count
0,10000


## 🔑 Keys Setup (in SQL Database)

In [ ]:
key_setup = [
    # PRIMARY KEY
    """
    ALTER TABLE top_artists
    ADD CONSTRAINT pk_top_artists PRIMARY KEY (name)
    """,

    # FOREIGN KEY
    """
    ALTER TABLE top_songs
    ADD CONSTRAINT fk_songs_artist
    FOREIGN KEY (artist)
    REFERENCES top_artists(name)
    """,

    # INDEX
    """
    CREATE INDEX IF NOT EXISTS idx_songs_artist_name
    ON top_songs(artist)
    """
]

with engine.begin() as conn:
    for stmt in key_setup:
        conn.execute(text(stmt))